# 01 — Ingest Customer Events to Bronze

## Purpose

This notebook incrementally ingests raw customer snapshots and CDC events from the canonical CRM landing path into an append-only Bronze Delta table using Databricks Auto Loader.

The Bronze layer preserves source values without applying business transformations. Technical metadata is added for file lineage, ingestion auditing, schema drift detection, duplicate analysis, and downstream CDC processing.

## Business Grain

One row represents one raw customer source event.

A customer can appear multiple times because Bronze preserves the initial INSERT event and all subsequent INSERT, UPDATE, or DELETE events.

## Ingestion Design

- Explicit customer event schema
- Incremental JSON ingestion with Auto Loader
- Checkpoint-based exactly-once file processing
- `AvailableNow` trigger for Serverless compatibility
- Rescued-data capture for unexpected fields
- Source-file and ingestion metadata
- Deterministic record hashes
- Source-to-Bronze reconciliation
- Idempotent rerun validation

## Source

- `/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers`

## Target

- `workspace.revenue_leakage_bronze.customer_events`

## Expected First-Load Volume

- 5,000 initial customer events
- 500 CDC events
- 5,500 total Bronze events

## 1. Configuration and Explicit Source Schema

Define the canonical CRM source, Auto Loader state locations, Bronze target, expected volumes, and customer event data contract.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
)

SOURCE_SYSTEM = "CRM"
SOURCE_ENTITY = "customers"
SOURCE_FORMAT = "json"

EXPECTED_INITIAL_CUSTOMER_COUNT = 5000
EXPECTED_CUSTOMER_CDC_COUNT = 500
EXPECTED_TOTAL_CUSTOMER_EVENT_COUNT = 5500

EXPECTED_OPERATION_COUNTS = {
    "INSERT": 5200,
    "UPDATE": 250,
    "DELETE": 50,
}

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/landing"
)

CUSTOMERS_SOURCE_PATH = (
    f"{LANDING_PATH}/crm/customers"
)

CUSTOMERS_INITIAL_PATH = (
    f"{CUSTOMERS_SOURCE_PATH}/initial_load"
)

CUSTOMERS_CHANGE_BATCH_PATH = (
    f"{CUSTOMERS_SOURCE_PATH}/change_batch_001"
)

CUSTOMERS_SCHEMA_PATH = (
    f"{LANDING_PATH}/_schemas/"
    "bronze/crm/customers"
)

CUSTOMERS_CHECKPOINT_PATH = (
    f"{LANDING_PATH}/_checkpoints/"
    "bronze/crm/customers"
)

CUSTOMERS_BRONZE_TABLE = (
    "workspace.revenue_leakage_bronze."
    "customer_events"
)

CUSTOMER_EVENT_SCHEMA = StructType([
    StructField(
        "customer_id",
        StringType(),
        False
    ),
    StructField(
        "first_name",
        StringType(),
        False
    ),
    StructField(
        "last_name",
        StringType(),
        False
    ),
    StructField(
        "email",
        StringType(),
        True
    ),
    StructField(
        "country",
        StringType(),
        False
    ),
    StructField(
        "region",
        StringType(),
        False
    ),
    StructField(
        "customer_segment",
        StringType(),
        False
    ),
    StructField(
        "signup_date",
        DateType(),
        True
    ),
    StructField(
        "customer_status",
        StringType(),
        False
    ),
    StructField(
        "operation",
        StringType(),
        False
    ),
    StructField(
        "event_timestamp",
        TimestampType(),
        True
    ),
])

CUSTOMER_SOURCE_COLUMNS = (
    CUSTOMER_EVENT_SCHEMA.fieldNames()
)

BRONZE_METADATA_COLUMNS = [
    "_source_system",
    "_source_entity",
    "_source_file_path",
    "_source_file_name",
    "_source_file_size",
    "_source_file_modification_time",
    "_ingested_at",
    "_ingestion_date",
    "_record_hash",
    "_rescued_data",
]

CUSTOMER_BRONZE_COLUMNS = (
    CUSTOMER_SOURCE_COLUMNS
    + BRONZE_METADATA_COLUMNS
)

## 2. Validate Canonical CRM Landing Files

Load the initial customer snapshot and CDC batch using the explicit source schema.

Fail-fast controls verify source accessibility, expected row volumes, required fields, event uniqueness, customer identifier formats, lifecycle domains, timestamps, and operation distributions before Auto Loader writes any Bronze data.

In [0]:
customers_initial_source_df = (
    spark.read
    .schema(CUSTOMER_EVENT_SCHEMA)
    .json(CUSTOMERS_INITIAL_PATH)
    .withColumn(
        "_landing_batch",
        F.lit("initial_load")
    )
)

customers_changes_source_df = (
    spark.read
    .schema(CUSTOMER_EVENT_SCHEMA)
    .json(CUSTOMERS_CHANGE_BATCH_PATH)
    .withColumn(
        "_landing_batch",
        F.lit("change_batch_001")
    )
)

customers_landing_source_df = (
    customers_initial_source_df
    .unionByName(customers_changes_source_df)
)

initial_source_count = (
    customers_initial_source_df.count()
)

customer_cdc_source_count = (
    customers_changes_source_df.count()
)

total_landing_source_count = (
    customers_landing_source_df.count()
)

distinct_source_event_count = (
    customers_landing_source_df
    .select(
        "customer_id",
        "operation",
        "event_timestamp"
    )
    .distinct()
    .count()
)

source_null_condition = F.lit(False)

for column_name in CUSTOMER_SOURCE_COLUMNS:
    source_null_condition = (
        source_null_condition
        | F.col(column_name).isNull()
    )

null_source_field_count = (
    customers_landing_source_df
    .filter(source_null_condition)
    .count()
)

duplicate_source_event_count = (
    customers_landing_source_df
    .groupBy(
        "customer_id",
        "operation",
        "event_timestamp"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_initial_customer_count = (
    customers_initial_source_df
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

invalid_customer_id_count = (
    customers_landing_source_df
    .filter(
        ~F.col("customer_id").rlike(
            r"^C[0-9]{6}$"
        )
    )
    .count()
)

invalid_source_domain_count = (
    customers_landing_source_df
    .filter(
        ~F.col("operation").isin(
            "INSERT",
            "UPDATE",
            "DELETE"
        )
        | ~F.col("customer_status").isin(
            "Active",
            "Inactive"
        )
        | ~F.col("customer_segment").isin(
            "Standard",
            "Premium",
            "VIP"
        )
        | ~F.col("email").contains("@")
        | (
            F.length(
                F.trim(F.col("country"))
            ) == 0
        )
        | (
            F.length(
                F.trim(F.col("region"))
            ) == 0
        )
    )
    .count()
)

invalid_source_timestamp_count = (
    customers_landing_source_df
    .filter(
        F.col("event_timestamp").cast("date")
        < F.col("signup_date")
    )
    .count()
)

initial_non_insert_count = (
    customers_initial_source_df
    .filter(
        F.col("operation") != "INSERT"
    )
    .count()
)

actual_operation_counts = {
    row["operation"]: row["count"]
    for row in (
        customers_landing_source_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

assert (
    initial_source_count
    == EXPECTED_INITIAL_CUSTOMER_COUNT
), (
    f"Expected {EXPECTED_INITIAL_CUSTOMER_COUNT:,} "
    f"initial customers, but found "
    f"{initial_source_count:,}."
)

assert (
    customer_cdc_source_count
    == EXPECTED_CUSTOMER_CDC_COUNT
), (
    f"Expected {EXPECTED_CUSTOMER_CDC_COUNT:,} "
    f"CDC events, but found "
    f"{customer_cdc_source_count:,}."
)

assert (
    total_landing_source_count
    == EXPECTED_TOTAL_CUSTOMER_EVENT_COUNT
), "Total customer source count is incorrect."

assert (
    distinct_source_event_count
    == total_landing_source_count
), "Duplicate customer event keys were detected."

assert null_source_field_count == 0, (
    "Null required customer source fields were detected."
)

assert duplicate_source_event_count == 0, (
    "Duplicate customer source events were detected."
)

assert duplicate_initial_customer_count == 0, (
    "Duplicate initial customer IDs were detected."
)

assert invalid_customer_id_count == 0, (
    "Invalid customer identifier formats were detected."
)

assert invalid_source_domain_count == 0, (
    "Invalid customer source-domain values were detected."
)

assert invalid_source_timestamp_count == 0, (
    "Customer events before signup were detected."
)

assert initial_non_insert_count == 0, (
    "The initial snapshot contains non-INSERT events."
)

assert actual_operation_counts == (
    EXPECTED_OPERATION_COUNTS
), (
    "Customer operation counts do not match "
    "the expected distribution."
)

print(
    f"Initial customer events: "
    f"{initial_source_count:,}"
)

print(
    f"Customer CDC events: "
    f"{customer_cdc_source_count:,}"
)

print(
    f"Total landing events: "
    f"{total_landing_source_count:,}"
)

print(
    f"Distinct source event keys: "
    f"{distinct_source_event_count:,}"
)

print(
    f"Null required source fields: "
    f"{null_source_field_count:,}"
)

print(
    f"Duplicate source events: "
    f"{duplicate_source_event_count:,}"
)

print(
    f"Duplicate initial customer IDs: "
    f"{duplicate_initial_customer_count:,}"
)

print(
    f"Invalid customer IDs: "
    f"{invalid_customer_id_count:,}"
)

print(
    f"Invalid source-domain values: "
    f"{invalid_source_domain_count:,}"
)

print(
    f"Invalid source timestamps: "
    f"{invalid_source_timestamp_count:,}"
)

display(
    customers_landing_source_df
    .groupBy(
        "_landing_batch",
        "operation"
    )
    .count()
    .orderBy(
        "_landing_batch",
        "operation"
    )
)

## 3. Incrementally Ingest Customer Events with Auto Loader

Read all available customer JSON files from the canonical CRM landing path and append them to the Bronze Delta table.

Auto Loader records processed files in its checkpoint. The `AvailableNow` trigger processes the current backlog and then stops, providing incremental and idempotent ingestion on Serverless compute.

Only technical metadata is added. Source business values remain unchanged.

In [0]:
customer_record_struct = F.struct(
    *[
        F.col(column_name)
        for column_name in CUSTOMER_SOURCE_COLUMNS
    ]
)

customer_events_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option(
        "cloudFiles.format",
        SOURCE_FORMAT
    )
    .option(
        "cloudFiles.schemaLocation",
        CUSTOMERS_SCHEMA_PATH
    )
    .option(
        "cloudFiles.schemaEvolutionMode",
        "rescue"
    )
    .option(
        "cloudFiles.includeExistingFiles",
        "true"
    )
    .option(
        "rescuedDataColumn",
        "_rescued_data"
    )
    .schema(CUSTOMER_EVENT_SCHEMA)
    .load(CUSTOMERS_SOURCE_PATH)
    .select(
        *[
            F.col(column_name)
            for column_name in CUSTOMER_SOURCE_COLUMNS
        ],

        F.lit(SOURCE_SYSTEM)
        .alias("_source_system"),

        F.lit(SOURCE_ENTITY)
        .alias("_source_entity"),

        F.col("_metadata.file_path")
        .alias("_source_file_path"),

        F.col("_metadata.file_name")
        .alias("_source_file_name"),

        F.col("_metadata.file_size")
        .alias("_source_file_size"),

        F.col("_metadata.file_modification_time")
        .alias("_source_file_modification_time"),

        F.current_timestamp()
        .alias("_ingested_at"),

        F.current_date()
        .alias("_ingestion_date"),

        F.sha2(
            F.to_json(customer_record_struct),
            256
        ).alias("_record_hash"),

        F.col("_rescued_data")
        .alias("_rescued_data")
    )
)

customer_ingestion_query = (
    customer_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CUSTOMERS_CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(CUSTOMERS_BRONZE_TABLE)
)

customer_ingestion_query.awaitTermination()

spark.sql(
    f"""
    COMMENT ON TABLE {CUSTOMERS_BRONZE_TABLE}
    IS
    'Append-only Bronze customer events ingested from the canonical CRM landing zone'
    """
)

spark.sql(
    f"""
    ALTER TABLE {CUSTOMERS_BRONZE_TABLE}
    SET TBLPROPERTIES (
        'layer' = 'bronze',
        'source_system' = 'CRM',
        'source_entity' = 'customers',
        'data_classification' = 'synthetic_customer_data'
    )
    """
)

print("Customer Auto Loader ingestion completed.")

print(
    f"Bronze table: "
    f"{CUSTOMERS_BRONZE_TABLE}"
)

print(
    f"Canonical source: "
    f"{CUSTOMERS_SOURCE_PATH}"
)

print(
    f"Checkpoint: "
    f"{CUSTOMERS_CHECKPOINT_PATH}"
)

## 4. Validate Bronze Integrity and Idempotency

Validate the completed Bronze ingestion against the canonical CRM landing source.

The controls verify row preservation, event uniqueness, record-hash uniqueness, metadata completeness, source lineage, rescued data, operation distributions, source-to-Bronze equality, Delta format, and checkpoint-based idempotency.

The ingestion is executed a second time with the same checkpoint. A correct incremental pipeline must add zero duplicate rows.

In [0]:
bronze_customers_df = (
    spark.table(CUSTOMERS_BRONZE_TABLE)
)

bronze_event_count = (
    bronze_customers_df.count()
)

distinct_bronze_event_count = (
    bronze_customers_df
    .select(
        "customer_id",
        "operation",
        "event_timestamp"
    )
    .distinct()
    .count()
)

distinct_record_hash_count = (
    bronze_customers_df
    .select("_record_hash")
    .distinct()
    .count()
)

bronze_source_null_condition = F.lit(False)

for column_name in CUSTOMER_SOURCE_COLUMNS:
    bronze_source_null_condition = (
        bronze_source_null_condition
        | F.col(column_name).isNull()
    )

bronze_null_source_field_count = (
    bronze_customers_df
    .filter(bronze_source_null_condition)
    .count()
)

REQUIRED_BRONZE_METADATA_COLUMNS = [
    column_name
    for column_name in BRONZE_METADATA_COLUMNS
    if column_name != "_rescued_data"
]

bronze_metadata_null_condition = F.lit(False)

for column_name in REQUIRED_BRONZE_METADATA_COLUMNS:
    bronze_metadata_null_condition = (
        bronze_metadata_null_condition
        | F.col(column_name).isNull()
    )

bronze_null_metadata_count = (
    bronze_customers_df
    .filter(bronze_metadata_null_condition)
    .count()
)

rescued_data_row_count = (
    bronze_customers_df
    .filter(
        F.col("_rescued_data").isNotNull()
    )
    .count()
)

invalid_lineage_metadata_count = (
    bronze_customers_df
    .filter(
        (
            F.col("_source_system")
            != SOURCE_SYSTEM
        )
        | (
            F.col("_source_entity")
            != SOURCE_ENTITY
        )
        | ~F.col("_source_file_path").contains(
            "/crm/customers/"
        )
        | (
            F.col("_source_file_size") <= 0
        )
        | (
            F.col("_ingestion_date")
            != F.col("_ingested_at").cast("date")
        )
    )
    .count()
)

bronze_initial_event_count = (
    bronze_customers_df
    .filter(
        F.col("_source_file_path").contains(
            "/initial_load/"
        )
    )
    .count()
)

bronze_cdc_event_count = (
    bronze_customers_df
    .filter(
        F.col("_source_file_path").contains(
            "/change_batch_001/"
        )
    )
    .count()
)

bronze_operation_counts = {
    row["operation"]: row["count"]
    for row in (
        bronze_customers_df
        .groupBy("operation")
        .count()
        .collect()
    )
}

source_to_bronze_mismatch_count = (
    customers_landing_source_df
    .select(*CUSTOMER_SOURCE_COLUMNS)
    .exceptAll(
        bronze_customers_df
        .select(*CUSTOMER_SOURCE_COLUMNS)
    )
    .count()
    + bronze_customers_df
    .select(*CUSTOMER_SOURCE_COLUMNS)
    .exceptAll(
        customers_landing_source_df
        .select(*CUSTOMER_SOURCE_COLUMNS)
    )
    .count()
)

customer_table_detail = (
    spark.sql(
        f"DESCRIBE DETAIL {CUSTOMERS_BRONZE_TABLE}"
    )
    .first()
)

customer_table_format = (
    customer_table_detail["format"]
)

assert (
    bronze_event_count
    == EXPECTED_TOTAL_CUSTOMER_EVENT_COUNT
), (
    f"Expected {EXPECTED_TOTAL_CUSTOMER_EVENT_COUNT:,} "
    f"Bronze events, but found "
    f"{bronze_event_count:,}."
)

assert (
    distinct_bronze_event_count
    == bronze_event_count
), "Duplicate Bronze event keys were detected."

assert (
    distinct_record_hash_count
    == bronze_event_count
), "Duplicate Bronze record hashes were detected."

assert bronze_null_source_field_count == 0, (
    "Null required source fields were detected."
)

assert bronze_null_metadata_count == 0, (
    "Null required Bronze metadata was detected."
)

assert rescued_data_row_count == 0, (
    "Unexpected source fields or type mismatches "
    "were captured in rescued data."
)

assert invalid_lineage_metadata_count == 0, (
    "Invalid canonical CRM lineage was detected."
)

assert (
    bronze_initial_event_count
    == EXPECTED_INITIAL_CUSTOMER_COUNT
), "Bronze initial-load count is incorrect."

assert (
    bronze_cdc_event_count
    == EXPECTED_CUSTOMER_CDC_COUNT
), "Bronze CDC count is incorrect."

assert (
    bronze_operation_counts
    == EXPECTED_OPERATION_COUNTS
), "Bronze operation distribution is incorrect."

assert source_to_bronze_mismatch_count == 0, (
    "Canonical source and Bronze contents do not match."
)

assert customer_table_format.lower() == "delta", (
    "The Bronze target is not a Delta table."
)

print(
    f"Bronze customer events: "
    f"{bronze_event_count:,}"
)

print(
    f"Distinct Bronze event keys: "
    f"{distinct_bronze_event_count:,}"
)

print(
    f"Distinct record hashes: "
    f"{distinct_record_hash_count:,}"
)

print(
    f"Null required source fields: "
    f"{bronze_null_source_field_count:,}"
)

print(
    f"Null required metadata fields: "
    f"{bronze_null_metadata_count:,}"
)

print(
    f"Rescued-data rows: "
    f"{rescued_data_row_count:,}"
)

print(
    f"Invalid canonical lineage rows: "
    f"{invalid_lineage_metadata_count:,}"
)

print(
    f"Initial-load events: "
    f"{bronze_initial_event_count:,}"
)

print(
    f"CDC events: "
    f"{bronze_cdc_event_count:,}"
)

print(
    f"Source/Bronze content mismatches: "
    f"{source_to_bronze_mismatch_count:,}"
)

print(
    f"Target format: "
    f"{customer_table_format}"
)

event_count_before_rerun = (
    spark.table(CUSTOMERS_BRONZE_TABLE)
    .count()
)

customer_idempotency_query = (
    customer_events_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CUSTOMERS_CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .toTable(CUSTOMERS_BRONZE_TABLE)
)

customer_idempotency_query.awaitTermination()

event_count_after_rerun = (
    spark.table(CUSTOMERS_BRONZE_TABLE)
    .count()
)

rows_added_during_rerun = (
    event_count_after_rerun
    - event_count_before_rerun
)

assert rows_added_during_rerun == 0, (
    "The Auto Loader rerun created duplicate rows."
)

print(
    f"Rows before idempotency rerun: "
    f"{event_count_before_rerun:,}"
)

print(
    f"Rows after idempotency rerun: "
    f"{event_count_after_rerun:,}"
)

print(
    f"Rows added during rerun: "
    f"{rows_added_during_rerun:,}"
)

display(
    bronze_customers_df
    .groupBy("operation")
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct(
            "customer_id"
        ).alias("distinct_customers"),
        F.min(
            "event_timestamp"
        ).alias("first_event_timestamp"),
        F.max(
            "event_timestamp"
        ).alias("last_event_timestamp")
    )
    .orderBy("operation")
)